In [138]:
import refinitiv.data as rd
import pandas as pd
import numpy as np

rd.open_session()

<refinitiv.data.session.Definition object at 0x118a550a0 {name='workspace'}>

In [139]:
# 10 satunnaista osaketta STOXX 600 universumista
universe = pd.read_csv("../../data/stoxx600_universe.csv")
STOCK = universe["RIC"].sample(10, random_state=42).tolist()
print("Osakkeet:", STOCK)

PARAMS = {"SDate": "2024-01-01", "EDate": "2025-12-31", "Frq": "D", "Curn": "EUR"}

Osakkeet: ['BOLL.PA', 'RACE.MI', 'VAR.OL', 'BASFn.DE', 'ENAG.MC', 'INPST.AS', 'ABVX.PA', 'SEBa.ST', 'BATS.L', 'MOBN.S']


In [140]:
df = rd.get_data(
    universe=STOCK,
    fields=[
        "TR.PriceClose.date",
        "TR.PriceClose",
        "TR.CompanyMarketCap",
        "TR.SharesOutstanding",
        "TR.PriceToCFPerShare",
        "TR.F.CF",
        "TR.F.NetCashFlowOp",
        "TR.CFPSActValue",
        "TR.F.PriceToCFPerShr",
        "TR.F.NetCFOpPerShr",
    ],
    parameters=PARAMS
)

# Instrument-sarake tulee aina kun haetaan useampi osake
print(f"Sarakkeet ({len(df.columns)}):", df.columns.tolist())
print(f"{len(df)} riviä, {df['Instrument'].nunique()} osaketta")
df.head()

Sarakkeet (11): ['Instrument', 'Date', 'Price Close', 'Company Market Cap', 'Outstanding Shares', 'Price To Cash Flow Per Share (Daily Time Series Ratio)', 'Cash Flow', 'Net Cash Flow from Operating Activities', 'Cash Flow Per Share - Actual', 'Price to Cash Flow per Share', 'Cash Flow from Operations per Share']
5062 riviä, 10 osaketta


/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to

,Instrument,Date,Price Close,Company Market Cap,Outstanding Shares,Price To Cash Flow Per Share (Daily Time Series Ratio),Cash Flow,Net Cash Flow from Operating Activities,Cash Flow Per Share - Actual,Price to Cash Flow per Share,Cash Flow from Operations per Share
0,BOLL.PA,2024-01-02,5.695,16806938059.929899,2932673610,15.376888,50000000.0,1164500000.0,0.414,306.625085,0.39649
1,BOLL.PA,2024-01-03,5.64,16644623469.360001,2932673610,15.228384,50000000.0,1164500000.0,0.414,306.625085,0.39649
2,BOLL.PA,2024-01-04,5.715,16865961547.409901,2932673610,15.430889,50000000.0,1164500000.0,0.414,306.625085,0.39649
3,BOLL.PA,2024-01-05,5.72,16880717419.2799,2932673610,15.444389,50000000.0,1164500000.0,0.414,306.625085,0.39649
4,BOLL.PA,2024-01-08,5.78,17057787881.719999,2932673610,15.606393,50000000.0,1164500000.0,0.414,306.625085,0.39649


In [ ]:
# Tyyppimuunnokset
cols = df.columns.tolist()
date_col = cols[1]  # "Date" on toinen sarake, Instrument on ensimmäinen
df[date_col] = pd.to_datetime(df[date_col])
for c in cols[2:]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Lyhytnimet viittauksiin
Price = df["Price Close"]
MktCap = df["Company Market Cap"]
Shares = df["Outstanding Shares"]
CF = df.groupby("Instrument")["Cash Flow"].ffill()
OpCF = df.groupby("Instrument")["Net Cash Flow from Operating Activities"].ffill()
CFPS_Act = df.groupby("Instrument")["Cash Flow Per Share - Actual"].ffill()
OpCFPS = df.groupby("Instrument")["Cash Flow from Operations per Share"].ffill()
Shares_ff = df.groupby("Instrument")["Outstanding Shares"].ffill()

# Itselasketut P/CF
df["CALC_PCF_MktCap_div_CF"]        = MktCap / CF #ei
df["CALC_PCF_MktCap_div_OpCF"]      = MktCap / OpCF # joo
df["CALC_PCF_Price_div_CF_Shares"]   = Price / (CF / Shares_ff) # ei
df["CALC_PCF_Price_div_OpCF_Shares"] = Price / (OpCF / Shares_ff) #joo
df["CALC_PCF_Price_div_CFPS_Act"]    = Price / CFPS_Act # joo
df["CALC_PCF_Price_div_OpCFPS"]      = Price / OpCFPS # joo

df.to_csv("pcf.csv", index=False)
print(f"pcf.csv ({len(df)} riviä)")
df

pcf.csv (5062 riviä)


,Instrument,Date,Price Close,Company Market Cap,Outstanding Shares,Price To Cash Flow Per Share (Daily Time Series Ratio),Cash Flow,Net Cash Flow from Operating Activities,Cash Flow Per Share - Actual,Price to Cash Flow per Share,Cash Flow from Operations per Share,CALC_PCF_MktCap_div_CF,CALC_PCF_MktCap_div_OpCF,CALC_PCF_Price_div_CF_Shares,CALC_PCF_Price_div_OpCF_Shares,CALC_PCF_Price_div_CFPS_Act,CALC_PCF_Price_div_OpCFPS
0,BOLL.PA,2024-01-02,5.695,16806938059.929899,2932673610,15.376888,50000000.0,1164500000.0,0.414,306.625085,0.39649,336.138761,14.432751,334.031524,14.342272,13.756039,14.363538
1,BOLL.PA,2024-01-03,5.64,16644623469.360001,2932673610,15.228384,50000000.0,1164500000.0,0.414,306.625085,0.39649,332.892469,14.293365,330.805583,14.203761,13.623188,14.22482
2,BOLL.PA,2024-01-04,5.715,16865961547.409901,2932673610,15.430889,50000000.0,1164500000.0,0.414,306.625085,0.39649,337.319231,14.483436,335.204594,14.39264,13.804348,14.41398
3,BOLL.PA,2024-01-05,5.72,16880717419.2799,2932673610,15.444389,50000000.0,1164500000.0,0.414,306.625085,0.39649,337.614348,14.496108,335.497861,14.405232,13.816425,14.426591
4,BOLL.PA,2024-01-08,5.78,17057787881.719999,2932673610,15.606393,50000000.0,1164500000.0,0.414,306.625085,0.39649,341.155758,14.648165,339.017069,14.556336,13.961353,14.577919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5057,MOBN.S,2025-12-19,388.285186,2892694738.61009,7255211,27.960995,138182684.634039,105981416.105531,17.620047,16.379365,14.805361,20.933844,27.294358,20.386715,26.58099,22.036558,26.225986
5058,MOBN.S,2025-12-22,391.88649,2919524174.48775,7255211,28.231711,138182684.634039,105981416.105531,17.620047,16.379365,14.805361,21.128003,27.547511,20.575799,26.827526,22.240945,26.469229
5059,MOBN.S,2025-12-23,389.710741,2903315016.39683,7255211,27.999669,138182684.634039,105981416.105531,17.620047,16.379365,14.805361,21.010701,27.394567,20.461563,26.67858,22.117464,26.322272
5060,MOBN.S,2025-12-29,390.919891,2912323086.92194,7255211,28.077016,138182684.634039,105981416.105531,17.620047,16.379365,14.805361,21.07589,27.479564,20.525048,26.761355,22.186087,26.403942


In [142]:
# Haetaan kvartaali-CF
q_cf = rd.get_data(
    universe=STOCK,
    fields=["TR.F.CF.periodenddate", "TR.F.CF"],
    parameters={"Period": "FQ-19:FQ0", "Frq": "FQ", "Curn": "EUR"}
)
q_cf.columns = ["Instrument", "QDate", "CF_Q"]

q_opcf = rd.get_data(
    universe=STOCK,
    fields=["TR.F.NetCashFlowOp.periodenddate", "TR.F.NetCashFlowOp"],
    parameters={"Period": "FQ-19:FQ0", "Frq": "FQ", "Curn": "EUR"}
)
q_opcf.columns = ["Instrument", "QDate", "OpCF_Q"]

# Kattavuus
for name, qdf in [("CF_Q", q_cf), ("OpCF_Q", q_opcf)]:
    for stock in STOCK:
        sub = qdf[qdf.Instrument == stock]
        n = pd.to_numeric(sub.iloc[:, -1], errors="coerce").notna().sum()
        print(f"  {stock} {name}: {n}/20 kvartaalia")
    print()

/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


  BOLL.PA CF_Q: 0/20 kvartaalia
  RACE.MI CF_Q: 20/20 kvartaalia
  VAR.OL CF_Q: 20/20 kvartaalia
  BASFn.DE CF_Q: 20/20 kvartaalia
  ENAG.MC CF_Q: 20/20 kvartaalia
  INPST.AS CF_Q: 20/20 kvartaalia
  ABVX.PA CF_Q: 3/20 kvartaalia
  SEBa.ST CF_Q: 20/20 kvartaalia
  BATS.L CF_Q: 0/20 kvartaalia
  MOBN.S CF_Q: 0/20 kvartaalia

  BOLL.PA OpCF_Q: 0/20 kvartaalia
  RACE.MI OpCF_Q: 20/20 kvartaalia
  VAR.OL OpCF_Q: 20/20 kvartaalia
  BASFn.DE OpCF_Q: 20/20 kvartaalia
  ENAG.MC OpCF_Q: 20/20 kvartaalia
  INPST.AS OpCF_Q: 19/20 kvartaalia
  ABVX.PA OpCF_Q: 2/20 kvartaalia
  SEBa.ST OpCF_Q: 20/20 kvartaalia
  BATS.L OpCF_Q: 0/20 kvartaalia
  MOBN.S OpCF_Q: 0/20 kvartaalia



/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [143]:
# Lasketaan TTM CF ja OpCF, yhdistetään päivädataan
# Käytetään vain osakkeita joilla kvartaalidata on saatavilla

def make_ttm(q_df, val_col, ttm_col):
    q = q_df.copy()
    q["QDate"] = pd.to_datetime(q["QDate"])
    q[val_col] = pd.to_numeric(q[val_col], errors="coerce")
    q = q.dropna(subset=["QDate", val_col]).sort_values(["Instrument", "QDate"])
    q[ttm_col] = q.groupby("Instrument")[val_col].rolling(4).sum().reset_index(level=0, drop=True)
    return q[["Instrument", "QDate", ttm_col]].dropna().rename(columns={"QDate": "Date"})

ttm_cf   = make_ttm(q_cf, "CF_Q", "TTM_CF")
ttm_opcf = make_ttm(q_opcf, "OpCF_Q", "TTM_OpCF")

# Merge per osake
df = df.sort_values(["Instrument", "Date"]).dropna(subset=["Date"])

parts = []
for stock in STOCK:
    d = df[df.Instrument == stock].copy().reset_index(drop=True)
    for ttm_df, col in [(ttm_cf, "TTM_CF"), (ttm_opcf, "TTM_OpCF")]:
        t = ttm_df[ttm_df.Instrument == stock][["Date", col]].copy().reset_index(drop=True)
        if len(t) > 0:
            d = pd.merge_asof(d, t, on="Date", direction="backward")
        else:
            d[col] = np.nan
    parts.append(d)

df = pd.concat(parts, ignore_index=True)

# Shares: MktCap / Price (aina saatavilla, ei tarvitse erillistä hakua)
df["Shares_calc"] = df["Company Market Cap"] / df["Price Close"]

# Kaikki P/CF variantit
df["CALC_PCF_MktCap_div_TTM_CF"]   = df["Company Market Cap"] / df["TTM_CF"]
df["CALC_PCF_MktCap_div_TTM_OpCF"] = df["Company Market Cap"] / df["TTM_OpCF"]
df["CALC_PCF_Price_div_TTM_CF_Shr"]  = df["Price Close"] / (df["TTM_CF"] / df["Shares_calc"])
df["CALC_PCF_Price_div_TTM_OpCF_Shr"] = df["Price Close"] / (df["TTM_OpCF"] / df["Shares_calc"])

# Vertailu: kuinka lähellä REF_PCF:ää? (vain rivit joissa molemmat ei-NaN)
ref = df["Price To Cash Flow Per Share (Daily Time Series Ratio)"]
for calc_col in ["CALC_PCF_MktCap_div_TTM_CF", "CALC_PCF_MktCap_div_TTM_OpCF",
                  "CALC_PCF_Price_div_TTM_CF_Shr", "CALC_PCF_Price_div_TTM_OpCF_Shr",
                  "CALC_PCF_MktCap_div_CF", "CALC_PCF_MktCap_div_OpCF",
                  "CALC_PCF_Price_div_CF_Shares", "CALC_PCF_Price_div_OpCF_Shares",
                  "CALC_PCF_Price_div_CFPS_Act", "CALC_PCF_Price_div_OpCFPS"]:
    mask = ref.notna() & df[calc_col].notna()
    if mask.sum() > 0:
        diff = ((df.loc[mask, calc_col] - ref[mask]) / ref[mask] * 100).abs().median()
        print(f"  {calc_col}: median |ero| = {diff:.1f}%")
    else:
        print(f"  {calc_col}: ei vertailtavia rivejä")

  CALC_PCF_MktCap_div_TTM_CF: median |ero| = 26.5%
  CALC_PCF_MktCap_div_TTM_OpCF: median |ero| = 4.7%
  CALC_PCF_Price_div_TTM_CF_Shr: median |ero| = 26.5%
  CALC_PCF_Price_div_TTM_OpCF_Shr: median |ero| = 4.7%
  CALC_PCF_MktCap_div_CF: median |ero| = 50.0%
  CALC_PCF_MktCap_div_OpCF: median |ero| = 13.7%
  CALC_PCF_Price_div_CF_Shares: median |ero| = 49.7%
  CALC_PCF_Price_div_OpCF_Shares: median |ero| = 11.9%
  CALC_PCF_Price_div_CFPS_Act: median |ero| = 9.2%
  CALC_PCF_Price_div_OpCFPS: median |ero| = 9.5%
